## Modules import, threads number setting

In [22]:
import os
os.environ["POLARS_MAX_THREADS"] = "8"

In [23]:
from pathlib import Path
import time
from typing import Optional, Dict, Callable

import polars as pl
import pyorc
import pyarrow.orc as orc
import sqlite3

## Formats writing time tests 

### Dataset info

The first half of dataset ["Official Reddit r/place Dataset CSV"](https://www.kaggle.com/datasets/antoinecarpentier/redditrplacecsv) is used in this notebook. The dataset is a table containing of records of Reddit users pixel coloring (tiling) flashmob together with moderators actions.

Dataset properties:
- files number - 1
- format - `.csv`
- dataset size - 21 Gb (therefore, it's half is ~10.5 Gb)
- table columns: `timestamp`, `userid`, `pixelcolor`, `coordinate`

Table properties:
- `timestamp` - the UTC time of the tile placement
- `userid` - a hashed identifier for each user placing the tile. These are not Reddit userids, but a hashed identifier to allow correlating tiles placed by the same user
- `pixel_color` - the hex color code of the tile placed
- `coordinate` - the `(x,y)` coordinate of the tile on the canvas (`(0,0)` is the top left corner, `(1999,0)` is the top right corner).

Moderator actions are instances of moderators using a rectangle drawing tool to handle inappropriate content. These rows differ in the coordinate tuple which contain four values instead of two - `(x1,y1,x2,y2)` corresponding to the upper left x1, y1 coordinate and the lower right x2, y2 coordinate of the moderation rect. These events apply the specified color to all tiles within those two points, inclusive.

### Dataset reading

In [24]:
dataset_name = "2022_place_canvas_history_half"
dataset_dir = Path("D:/Downloads/task_1_dataset")
formats_dir = Path("D:/Downloads/task_1_dataset/formats")
os.makedirs(formats_dir, exist_ok=True)

formats_benchmark = {}

In [25]:
dataset = pl.scan_csv(dataset_dir / f"{dataset_name}.csv", has_header=True, low_memory=True)

In [26]:
dataset.head(5).collect()

timestamp,user_id,pixel_color,coordinate
str,str,str,str
"""2022-04-04 00:53:51.577 UTC""","""ovTZk4GyTS1mDQnTbV+vDOCu1f+u6w…","""#00CCC0""","""826,1048"""
"""2022-04-04 00:53:53.758 UTC""","""6NSgFa1CvIPly1VniNhlbrmoN3vgDF…","""#94B3FF""","""583,1031"""
"""2022-04-04 00:53:54.685 UTC""","""O5Oityp3Z3owzTuwM9XnMggpLcqKEu…","""#6A5CFF""","""1873,558"""
"""2022-04-04 00:54:57.541 UTC""","""tc273UiqS0wKa6VwiOs/iz/t4LyPYr…","""#009EAA""","""1627,255"""
"""2022-04-04 00:55:16.307 UTC""","""OOWsU/HLb4UUkQwclDeXFtsJTOXMlA…","""#94B3FF""","""49,1478"""


In [16]:
print("Dataset columns names: ", dataset.collect_schema().names())
print("Dataset columns types: ", dataset.collect_schema().dtypes())

Dataset columns names:  ['timestamp', 'user_id', 'pixel_color', 'coordinate']
Dataset columns types:  [String, String, String, String]


### Writing dataset into chosen formats

#### Sinking & auxiliary functions

In [27]:
def jsonl_to_json(jsonl_path: Path,
                  json_path: Path):
    """
    Creates .json file out of .jsonl
    """
    with open(jsonl_path, "r") as jsonl_file, open(json_path, "w", encoding="utf-8") as json_file:
        json_file.write("[\n")
        
        is_first_line = True
        for line in jsonl_file:
            line = jsonl_file.readline()
            if not line:
                continue
            if not is_first_line:
                json_file.write(",\n")
            else: 
                is_first_line = False
            json_file.write(line)
        
        json_file.write("\n]")


def sink_json_via_jsonl(dataset_df: pl.LazyFrame, 
                        json_path: Path):
    jsonl_path = json_path.with_suffix(".jsonl")
    dataset_df.sink_ndjson(jsonl_path)
    jsonl_to_json(jsonl_path, json_path)


def merge_orcs(orcs_dir: Path,
               final_orc_path: Path):
    os.makedirs(final_orc_path.parent, exist_ok=True)

    orcs_list = os.listdir(orcs_dir)
    with open(orcs_list[0], "rb") as orc_file:
        schema = str(pyorc.Reader(orc_file).schema)

    with open(final_orc_path, "wb") as final_orc_file:
        writer = pyorc.Writer(final_orc_file, schema)
        for file_path in orcs_list:
            with open(file_path, "rb") as orc_file:
                reader = pyorc.Reader(orc_file)
                if str(reader.schema) == schema:
                    for row in reader:
                        writer.write(row)

def sink_orc_chunked(dataset_df: pl.LazyFrame, 
                     orc_path: Path, 
                     dataset_name: Optional[str] = "dataset",
                     chunk_size: Optional[int] = 100_000,
                     return_one_file: Optional[bool] = False):
    orcs_dir = orc_path.parent / "orcs_chunks"
    os.makedirs(orcs_dir, exist_ok=True)

    for i, dataset_chunk in enumerate(dataset_df.collect(engine="streaming").iter_slices(chunk_size)):
        arrow_table = dataset_chunk.to_arrow()
        orc.write_table(arrow_table, orcs_dir / f"{dataset_name}_chunk_{i}.orc")

    if return_one_file:
        merge_orcs(orcs_dir, orcs_dir)


def sink_sqlite(dataset_df: pl.LazyFrame, 
                path: Path, 
                table_name: Optional[str] = "dataset",
                chunk_size: Optional[int] = 100_000):
    conn = sqlite3.connect(path)
    chunks = dataset_df.collect(engine="streaming").iter_slices(chunk_size)
    first_chunk = True
    for chunk in chunks:
        chunk_pd = chunk.to_pandas()
        if first_chunk:
            chunk_pd.to_sql(table_name, conn, if_exists="replace", index=False, chunksize=chunk_size)
            first_chunk = False
        else:
            chunk_pd.to_sql(table_name, conn, if_exists="append", index=False, chunksize=chunk_size)
    conn.close()


def measure_writing_time(dataset_df: pl.LazyFrame, 
                         path: Path, 
                         write_func: Callable,
                         verbose: Optional[bool] = True, 
                         **kwargs) -> Dict:
    """Measures CPU & wall time on saving tabular dataset into file format"""
    start_wall = time.time()
    start_cpu = time.process_time()
    
    write_func(dataset_df, path, **kwargs)
    
    time_cpu = time.process_time() - start_cpu
    time_wall = time.time() - start_wall
    
    file_size_mb = os.path.getsize(path) / (1024 ** 2) if path.is_file() else "No file"

    if verbose:
        print("Writing finished in time:\n"
              f"    cpu:  {time_cpu:.3f} s\n"
              f"    wall: {time_wall:.3f} s")
    return {
        "cpu_time": time_cpu,
        "wall_time": time_wall,
        "file_size_mb": file_size_mb
    }

#### CSV-like formats

In [28]:
formats_args_csvs = [("csv", ","), ("tsv", "\t")]

for format, separator in formats_args_csvs:
    print("Writing dataset into format:", format)
    formats_benchmark[format] = measure_writing_time(
        dataset, 
        formats_dir / f"{dataset_name}.{format}", 
        pl.LazyFrame.sink_csv, 
        separator=separator
    )

Writing dataset into format: csv
Writing finished in time:
    cpu:  57.719 s
    wall: 63.555 s
Writing dataset into format: tsv
Writing finished in time:
    cpu:  55.906 s
    wall: 59.321 s


Результаты для разного числа потоков:

| Format  | CPU time (s) - 1 thread | CPU time (s) - 8 threads | Wall time (s) - 1 thread  | Wall time (s) - 8 threads |
|---------|-------------------------|--------------------------|---------------------------|---------------------------|
| csv     | 60.578                  | 57.719                   | 64.577                    | 63.555                    |
| tsv     | 59.234                  | 55.906                   | 60.759                    | 59.321                    |

#### Formats with sinking function without additional params: json, jsonl, parquet, orc

In [29]:
formats_funcs = [
    ("json", sink_json_via_jsonl),
    ("jsonl", pl.LazyFrame.sink_ndjson),
    ("parquet", pl.LazyFrame.sink_parquet), 
    ("orc", sink_orc_chunked)
]

for format, sink_func in formats_funcs:
    print("Writing dataset into format:", format)
    formats_benchmark[format] = measure_writing_time(
        dataset, 
        formats_dir / f"{dataset_name}.{format}", 
        sink_func
    )

Writing dataset into format: json
Writing finished in time:
    cpu:  216.891 s
    wall: 218.386 s
Writing dataset into format: jsonl
Writing finished in time:
    cpu:  72.750 s
    wall: 75.378 s
Writing dataset into format: parquet
Writing finished in time:
    cpu:  211.281 s
    wall: 220.877 s
Writing dataset into format: orc
Writing finished in time:
    cpu:  199.062 s
    wall: 2051.114 s


Результаты для разного числа потоков:

| Format  | CPU time (s) - 1 thread | CPU time (s) - 8 threads | Wall time (s) - 1 thread  | Wall time (s) - 8 threads |
|---------|-------------------------|--------------------------|---------------------------|---------------------------|
| json    | 231.109                 | 216.891                  | 232.84                    | 218.386                   |
| jsonl   | 82.984                  | 72.75                    | 84.404                    | 75.378                    |
| parquet | 255.828                 | 211.281                  | 264.868                   | 220.877                   |
| orc     | 207.5                   | 199.062                  | 352.188                   | 2051.114                  |

#### SQL format

In [30]:
formats_sqls = [
    ("sqlite", "db", sink_sqlite)
]

for format, extension, sink_func in formats_sqls:
    print("Writing dataset into format:", format)
    formats_benchmark[format] = measure_writing_time(
        dataset, 
        formats_dir / f"{dataset_name}.{extension}", 
        sink_func,
        table_name=dataset_name
    )

Writing dataset into format: sqlite
Writing finished in time:
    cpu:  475.453 s
    wall: 1400.996 s


Результаты для разного числа потоков:

| Format  | CPU time (s) - 1 thread | CPU time (s) - 8 threads | Wall time (s) - 1 threads | Wall time (s) - 8 threads |
|---------|-------------------------|--------------------------|---------------------------|---------------------------|
| sqlite  | 523.547                 | 475.453                  | 734.97                    | 1400.996                  |

#### Time results comparison

In [31]:
print(f"{'Format':<10} {'CPU time (s)':<15} {'Wall time (s)':<15} {'File size (MB)':<15}\n")
for format, result in formats_benchmark.items():
    print(f"{format:<10} {result['cpu_time']:<15.2f} {result['wall_time']:<15.2f} {result['file_size_mb']:<15}")

Format     CPU time (s)    Wall time (s)   File size (MB) 

csv        57.72           63.55           10330.189376831055
tsv        55.91           59.32           10177.264739990234
json       216.89          218.39          7497.195596694946
jsonl      72.75           75.38           14688.541487693787
parquet    211.28          220.88          5576.622200012207
orc        199.06          2051.11         No file        
sqlite     475.45          1401.00         11216.60546875 


Сравнение результатов для разного числа потоков:

| Format  | CPU time (s) - 1 thread | CPU time (s) - 8 threads | Wall time (s) - 1 thread  | Wall time (s) - 8 threads | File size (MB) |
|---------|-------------------------|--------------------------|---------------------------|---------------------------|----------------|
| csv     | 60.58                   | 57.72                    | 64.58                     | 63.55                     | 10330.189      |
| tsv     | 59.23                   | 55.91                    | 60.76                     | 59.32                     | 10177.265      |
| json    | 231.11                  | 216.89                   | 232.84                    | 218.39                    | 7497.196       |
| jsonl   | 82.98                   | 72.75                    | 84.40                     | 75.38                     | 14688.541      |
| parquet | 255.83                  | 211.28                   | 264.87                    | 220.88                    | 5576.622       |
| orc     | 207.50                  | 199.06                   | 352.19                    | 2051.11                   | 9961.311       |
| sqlite  | 523.55                  | 475.45                   | 734.97                    | 1401.00                   | 11216.605      |

## SQL requests

### Predicate filtering
Show & count the number of administrator interventions into tiling

In [20]:
dataset_admin_actions = dataset.filter(
    pl.col("coordinate").str.split(",").list.len() > 2
).collect()
dataset_admin_actions

timestamp,user_id,pixel_color,coordinate
str,str,str,str
"""2022-04-04 01:22:50.891 UTC""","""q+XjkQ6WRx0aBLtb2xRGWBrsHALXej…","""#000000""","""1349,1718,1424,1752"""
"""2022-04-03 23:29:52.139 UTC""","""gS0DWvPgaiQkHvG4NsHveLvpn8uf50…","""#FFB470""","""297,1750,364,1813"""
"""2022-04-03 23:05:04.703 UTC""","""m8NEcPbf5XRV5ppeuZ3KLIYAG8GuHk…","""#FFB470""","""298,1805,329,1839"""
"""2022-04-03 23:08:50.038 UTC""","""LKS2u3QL2N3Olv7rnUCWry4KJ5K4Ea…","""#FFB470""","""257,1736,296,1780"""
"""2022-04-03 23:03:29.93 UTC""","""q/Dk6lmcXm8bcDbNIhDglz7kFuCmX6…","""#FFB470""","""298,1770,334,1803"""
…,…,…,…
"""2022-04-04 15:58:33.239 UTC""","""vnCNr84RaVDsTzLffAYaiEcF/4WIDD…","""#94B3FF""","""551,1311,562,1342"""
"""2022-04-04 15:59:10.926 UTC""","""8dCm+y4XG/R21ijH9er0xcYdMU3lFa…","""#94B3FF""","""547,1330,550,1342"""
"""2022-04-01 14:44:08.158 UTC""","""UY7pWDspuiKPKlsEMYNjkKYnoLkwPW…","""#898D90""","""862,540,868,544"""


In [23]:
n_admins_actions = dataset_admin_actions.select(pl.len()).item()
n_actions = dataset.select(pl.len()).collect().item()

print(f"Admins' actions of participants': {n_admins_actions}/{n_actions}, "
      f"which is {n_admins_actions/n_actions:.10%}")

Admins' actions of participants': 19/80176552, which is 0.0000236977%


### Grouping + aggregation

1. calculate dictribution of coordinate string length (to ensure there are no coordinates of incorrect format)

In [15]:
coords_len_distr = (
    dataset
        .with_columns(pl.col("coordinate").str.split(",").list.len().alias("coord_len"))
        .group_by("coord_len")
        .agg(pl.len().alias("count"))
        .collect()
)
coords_len_distr

coord_len,count
u32,u32
2,80176533
4,19


2. find mean point of all users actions

In [ ]:
(dataset
    .with_columns(
        pl.col("coordinate").str.split(",").alias("coordinate_as_list")
    )
    .filter(
        pl.col("coordinate_as_list").list.len() == 2
    )
    .select(
        pl.col("coordinate_as_list").list.get(0).cast(pl.Float32).mean().alias("x_mean"),
        pl.col("coordinate_as_list").list.get(1).cast(pl.Float32).mean().alias("y_mean"),
    )
    .collect()
)

x_mean,y_mean
f16,f16
766.0,637.5


3. find mean tile coordinate for every user

In [30]:
dataset_users_mean_coords = ( dataset
    .with_columns(pl.col("coordinate").str.split(",").alias("coordinate_as_list"))
    .filter(pl.col("coordinate_as_list").list.len() == 2)
    .select(
        "user_id",
        pl.col("coordinate_as_list").list.get(0).cast(pl.Float32).alias("x"),
        pl.col("coordinate_as_list").list.get(1).cast(pl.Float32).alias("y"),
    )
    .group_by("user_id")
    .agg(
        pl.len().alias("n_operations"),
        pl.col("x").mean().alias("x_mean"),
        pl.col("y").mean().alias("y_mean"),
    )
)
dataset_users_mean_coords.head(10).collect()

C:\Users\Элина\AppData\Local\Temp\ipykernel_21280\3902353766.py:11: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  pl.count().alias("n_operations"),


user_id,n_operations,x_mean,y_mean
str,u32,f32,f32
"""5jYknx9BdxuEGL58FiDbPK1mkmpn0y…",1,1194.0,799.0
"""nfLGMIcLGpM01dYHSUQqX9shX0M7nh…",4,1147.5,314.0
"""RIBJsfnTfe6HAUQVOD502l0Ov5ioGc…",3,453.0,809.0
"""UyziaaXb9X3nZW8OelOyIlrK6PUbbW…",4,457.0,562.5
"""HhAJxRWx5E2pTYg0qlhuQIPPLPHh4o…",44,709.272705,907.863647
"""zT4k+tULFoxKKJJOV3Hkt3IvdjFFGA…",2,443.0,240.5
"""Wnn9k4oPTXCeBrxDF66ij162Tueqmq…",3,665.0,49.0
"""gizFAXZq0YDCVHRFA85Jo5Bpohz3Yr…",20,530.349976,570.950012
"""DdOqOBIV42t25WBfsByjaycaYrD7Cw…",8,962.75,629.25


### Window functions
Ranging + rolling mean to show rolling mean over coordinates for every 100 tiles

In [32]:
dataset_users_typed = (dataset
    .with_columns(
        pl.col("timestamp")
          .str.replace(" UTC", "")
          .str.to_datetime(time_zone="UTC")
    )
    .with_columns(
        pl.col("coordinate").str.split(",").alias("coordinate_as_list"))
    .filter(pl.col("coordinate_as_list").list.len() == 2)
    .select(
        "user_id",
        "timestamp",
        "pixel_color",
        pl.col("coordinate_as_list").list.get(0).cast(pl.Float32).alias("x"),
        pl.col("coordinate_as_list").list.get(1).cast(pl.Float32).alias("y"),
    )
    .sort("timestamp")
)
dataset_users_typed.head(5).collect()

user_id,timestamp,pixel_color,x,y
str,"datetime[μs, UTC]",str,f32,f32
"""lEjremCtNoQaJ6KGBSWsatGEMXwjqo…",2022-04-01 12:44:10.315 UTC,"""#7EED56""",42.0,42.0
"""Eicii64quWgAsZ9SmjGaKgs8FZhUxI…",2022-04-01 12:44:22.671 UTC,"""#00A368""",999.0,999.0
"""nkaugJr9j9Yn5dqcluwmWFQHdaLFNO…",2022-04-01 12:44:26.626 UTC,"""#3690EA""",44.0,42.0
"""jHYpQW0J/omQcvzImJ/rgoW/Ou+Rel…",2022-04-01 12:44:31.703 UTC,"""#D4D7D9""",2.0,2.0
"""4ndpsC1OEKlOZ61a+a/GacvxSqGtsb…",2022-04-01 12:44:44.409 UTC,"""#3690EA""",23.0,23.0


In [ ]:
window_size = 100

dataset_rolling_mean_color = ( dataset_users_typed
    .with_columns([
        pl.col("x")
          .rolling_mean(window_size=window_size, min_samples=1)
          .over(["user_id", "pixel_color"], order_by="timestamp")
          .alias(f"x_rolling_mean_{window_size}"),
        pl.col("y")
          .rolling_mean(window_size=window_size, min_samples=1)
          .over(["user_id", "pixel_color"], order_by="timestamp")
          .alias(f"y_rolling_mean_{window_size}"),
    ])
)
dataset_rolling_mean_color.head(10).collect()